# 14장 웹부록 | 우리 교실의 디지털 트윈 만들기

「영화로 보는 AI 리터러시」 14장 'AI가 만드는 또 하나의 세계: 홀로그램·메타버스·디지털 트윈'의 코랩 실습입니다.

**쓰는 법**

- 코드를 몰라도 괜찮습니다. 위에서부터 차례로 셀 왼쪽의 ▶(실행) 버튼을 누르세요.
- `# ← 이 숫자를 바꿔 보세요` 표시가 있는 곳만 고치고, 그 셀부터 다시 실행해 보세요.
- 처음부터 다시 하고 싶으면 메뉴에서 **런타임 → 모두 실행**을 누르세요.
- 따로 설치할 것은 없습니다. 그래프 글자는 영어로 나오고, 설명은 이 글에 우리말로 적었습니다.

**이 실습에서 할 일**

1. 가상의 온도 센서 4개가 보내는 값으로 교실의 **디지털 그림자**를 만듭니다.
2. 센서 기록으로 가상 교실(모델)을 진짜 교실에 맞춥니다.
3. 창문을 열거나 에어컨을 켰을 때 온도가 어떻게 바뀔지 **가상 교실에서 먼저** 시험합니다.
4. 시험 결과로 에어컨을 켜는 규칙을 고르고, 그 규칙이 다시 진짜 교실의 에어컨을 움직이게 하면 **디지털 트윈**이 완성됩니다.

책 3-6절 '현실의 쌍둥이: 디지털 트윈'을 함께 보세요. 이 실습의 '진짜 교실'도 컴퓨터 속에 만든 것입니다. 실제 센서가 있다면 그 값으로 바꿔 넣을 수 있습니다.

## 0단계. 실험에 쓸 '진짜 교실' 준비하기

진짜 교실과 센서가 없으니, 컴퓨터 속에 '진짜 교실' 역할을 할 교실을 하나 만들어 둡니다. 이 교실의 속사정(벽으로 열이 얼마나 빠지는지, 학생들이 열을 얼마나 내는지)은 **우리가 모른다고 치고**, 센서 값만 받아 보겠습니다.

교실을 위에서 내려다보면 네 구역으로 나뉘고, 구역마다 온도 센서가 하나씩 있습니다.

| | 창가 쪽 | 복도 쪽 |
|---|---|---|
| **앞(칠판, 에어컨)** | 센서 1 | 센서 2 |
| **뒤** | 센서 3 | 센서 4 |

- 여름날 9시부터 18시까지입니다. 바깥 온도는 아침 9시 24도쯤에서 오후 3시 32도까지 오릅니다. 교실은 어제의 열이 남아 아침에 28도쯤입니다.
- 수업은 9~12시, 13~17시이고, 학생 30명이 있으면 교실이 데워집니다.
- 오후에는 창가 쪽에 햇볕이 듭니다. 에어컨은 앞쪽에 있어서 앞이 더 빨리 시원해집니다.

이 셀은 실행만 하세요. 고칠 곳이 없습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

MINUTES = np.arange(540)                       # 9:00부터 1분 간격으로 540분(18:00까지)
CLOCK = 9 + MINUTES / 60
def hhmm(m): return f"{9 + m // 60:02d}:{m % 60:02d}"

def outdoor(hot=0.0):
    return 24 + hot + 8 * np.sin(np.pi * (CLOCK - 9) / 12)        # 바깥 온도
OCCUPIED = ((CLOCK >= 9) & (CLOCK < 12)) | ((CLOCK >= 13) & (CLOCK < 17))
SUN = np.where((CLOCK >= 12) & (CLOCK < 17), 1.0, 0.0)

WINDOW_SIDE = np.array([1, 0, 1, 0])           # 센서 1, 3이 창가
AC_SHARE = np.array([1.0, 1.0, 0.5, 0.5])      # 에어컨 바람은 앞쪽(1, 2)에 더 셉니다
NEIGHBORS = np.array([[0, 1, 1, 0], [1, 0, 0, 1], [1, 0, 0, 1], [0, 1, 1, 0]])

def classroom(wall_loss, people_heat, controller, t_out, start_temp=28.0):
    """교실 네 구역의 온도를 1분씩 계산합니다. 여러 교실(매개변수 묶음)을 한꺼번에 계산할 수 있습니다.
    controller(분, 지금 온도) → (창문 열림 0/1, 에어컨 켬 0/1)"""
    wall_loss = np.atleast_1d(wall_loss)[:, None]; people_heat = np.atleast_1d(people_heat)[:, None]
    T = np.full((len(wall_loss), 4), start_temp)
    temps, windows, acs = [], [], []
    for m in MINUTES:
        win, ac = controller(m, T)
        mix = 0.02 * (NEIGHBORS @ T.T).T - 0.02 * 2 * T                   # 이웃 구역과 섞임
        dT = (mix + wall_loss * (t_out[m] - T)                             # 벽으로 드나드는 열
              + win * 0.08 * WINDOW_SIDE * (t_out[m] - T)                  # 열린 창문
              + people_heat * OCCUPIED[m]                                  # 학생들의 체온
              + 0.02 * SUN[m] * WINDOW_SIDE                                # 오후 햇볕
              - ac * 0.03 * AC_SHARE * np.maximum(T - 22, 0))   # 에어컨(22도까지만 식힘)                                      # 에어컨
        T = T + dT
        temps.append(T.copy()); windows.append(win); acs.append(ac)
    return np.array(temps), np.array(windows), np.array(acs)

# '진짜 교실'의 속사정 (우리는 모른다고 칩니다)
_TRUE = dict(wall_loss=0.006, people_heat=0.012)

def real_classroom(controller, hot=0.0, seed=0):
    temps, w, a = classroom(_TRUE["wall_loss"], _TRUE["people_heat"], controller, outdoor(hot))
    noise = np.random.default_rng(seed).normal(0, 0.15, temps[:, 0].shape)   # 센서의 작은 오차
    return temps[:, 0] + noise, w, a

def old_habit(m, T):          # 지금까지의 습관: 창문은 닫고, 점심 뒤 13~17시에 에어컨
    return 0, int(13 <= CLOCK[m] < 17)

sensors, win_log, ac_log = real_classroom(old_habit, seed=1)
print("진짜 교실 준비 완료. 어제 하루(540분) 동안 센서 4개가 보낸 값을 받았습니다.")

## 1단계. 디지털 그림자: 센서 값을 받아 교실을 그리기

센서 4개가 1분마다 보내는 값으로 교실의 모습을 그립니다. 네 센서 사이의 온도는 가까운 센서 값을 섞어 채웁니다. 이것이 **디지털 그림자**입니다. 진짜 교실의 데이터가 자동으로 들어오기만 하고, 가상의 교실이 진짜 교실에 무언가를 하지는 않습니다.

👉 **바꿔 볼 곳**: `LOOK_AT`의 시각을 바꿔 보세요. 그 순간의 교실이 보입니다.

In [ ]:
LOOK_AT = "14:30"   # ← 이 시각을 바꿔 보세요 (09:00 ~ 17:59)

def to_minute(s):
    h, m = map(int, s.split(":")); return (h - 9) * 60 + m

def shadow_picture(ax, values, title):
    grid = np.array([[values[0], values[1]], [values[2], values[3]]])
    fine = np.zeros((40, 60))
    for i, y in enumerate(np.linspace(0, 1, 40)):              # 네 센서 값을 부드럽게 섞기
        for j, x in enumerate(np.linspace(0, 1, 60)):
            fine[i, j] = (grid[0, 0]*(1-x)*(1-y) + grid[0, 1]*x*(1-y) + grid[1, 0]*(1-x)*y + grid[1, 1]*x*y)
    im = ax.imshow(fine, cmap="coolwarm", vmin=24, vmax=34)
    for (y, x), v in zip([(3, 3), (3, 56), (36, 3), (36, 56)], values):
        ax.text(x, y, f"{v:.1f}C", ha="left" if x < 30 else "right", va="center", fontsize=9, weight="bold")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title)
    ax.set_xlabel("window side  <-->  hallway side"); ax.set_ylabel("back (bottom) / front, AC (top)")
    return im

m = to_minute(LOOK_AT)
fig, axes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [1, 1.6]})
im = shadow_picture(axes[0], sensors[m], f"Digital shadow at {LOOK_AT}")
plt.colorbar(im, ax=axes[0], shrink=0.8, label="temperature (C)")
for i in range(4):
    axes[1].plot(CLOCK, sensors[:, i], label=f"sensor {i+1}", lw=1)
axes[1].plot(CLOCK, outdoor(), "k--", label="outdoor")
axes[1].fill_between(CLOCK, 23, 36, where=ac_log > 0, color="tab:blue", alpha=0.08, label="AC on")
axes[1].axvline(9 + m / 60, color="gray"); axes[1].set_ylim(23, 36)
axes[1].set_xlabel("time of day (hour)"); axes[1].set_ylabel("temperature (C)")
axes[1].set_title("Yesterday's sensor data"); axes[1].legend(fontsize=8, ncol=3)
plt.tight_layout(); plt.show()
print(f"{LOOK_AT} 센서 값:", ", ".join(f"센서{i+1} {v:.1f}도" for i, v in enumerate(sensors[m])))

**읽는 법**: 오전에 학생들이 들어오면 온도가 오르고, 점심시간에는 조금 쉬었다가, 13시에 에어컨을 켜면 앞쪽(센서 1, 2)부터 내려갑니다. 창가(센서 1, 3)는 오후 햇볕 때문에 더 덥습니다.

## 2단계. 가상 교실을 진짜 교실에 맞추기

그림자만으로는 '만약 ~하면?'을 물을 수 없습니다. 그러려면 교실이 어떻게 데워지고 식는지 계산하는 **모델**이 필요합니다. 모델의 틀은 0단계와 같지만, 두 숫자는 모릅니다.

- **벽으로 열이 빠지는 정도** (`wall_loss`)
- **학생 30명이 내는 열** (`people_heat`)

여러 값을 넣어 어제 하루를 다시 계산해 보고, 센서 기록과 가장 비슷하게 나오는 값을 고릅니다. 3장의 학습처럼, 데이터에 가장 잘 맞는 숫자를 찾는 것입니다. 400가지 조합을 한꺼번에 계산합니다.

In [ ]:
wall_try = np.linspace(0.001, 0.015, 20)
people_try = np.linspace(0.002, 0.025, 20)
W_, P_ = np.meshgrid(wall_try, people_try)
replay = lambda m, T: (win_log[m], ac_log[m])            # 어제 실제로 했던 그대로(창문, 에어컨)
sim, _, _ = classroom(W_.ravel(), P_.ravel(), replay, outdoor())
error = np.sqrt(((sim - sensors[:, None, :]) ** 2).mean(axis=(0, 2)))   # 센서 기록과의 차이
best = np.argmin(error)
TWIN = dict(wall_loss=W_.ravel()[best], people_heat=P_.ravel()[best])
print(f"가장 잘 맞는 값: wall_loss = {TWIN['wall_loss']:.4f}, people_heat = {TWIN['people_heat']:.4f}")
print(f"센서 기록과의 평균 차이: {error[best]:.2f}도")

fit, _, _ = classroom(TWIN["wall_loss"], TWIN["people_heat"], replay, outdoor())
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].imshow(error.reshape(20, 20), origin="lower", aspect="auto", cmap="viridis_r",
               extent=[wall_try[0], wall_try[-1], people_try[0], people_try[-1]])
axes[0].plot(TWIN["wall_loss"], TWIN["people_heat"], "r*", ms=15)
axes[0].set_xlabel("wall_loss"); axes[0].set_ylabel("people_heat"); axes[0].set_title("Error for each guess (dark = good)")
for i in range(4):
    axes[1].plot(CLOCK, sensors[:, i], color=f"C{i}", lw=0.8, alpha=0.6)
    axes[1].plot(CLOCK, fit[:, 0, i], color=f"C{i}", lw=2, ls="--")
axes[1].set_title("Sensor data (thin) vs virtual classroom (dashed)")
axes[1].set_xlabel("time of day (hour)"); axes[1].set_ylabel("temperature (C)")
plt.tight_layout(); plt.show()

가상 교실(점선)이 센서 기록(가는 선)을 잘 따라갑니다. 이제 가상 교실은 진짜 교실의 **쌍둥이 후보**입니다.

## 3단계. 가상 교실에서 먼저 시험하기

내일은 오늘보다 2도 더 덥다는 예보가 나왔습니다. 진짜 교실에서 이것저것 시험하면 학생들이 더위에 시달리고 전기도 낭비합니다. 그래서 **가상 교실에서 먼저** 시험합니다.

- 아무것도 안 하기
- 하루 종일 창문 열기
- 아침(9~11시)에만 창문 열기
- 수업 시간 내내 에어컨 켜기
- 창문을 열고 에어컨도 켜기

**덥다**의 기준은 아래 `HOT_LINE`입니다. 수업 중에 네 구역 평균이 이 온도를 넘은 시간을 '더운 시간'으로 셉니다. 에어컨을 켠 시간은 전기 사용량을 나타냅니다.

👉 **바꿔 볼 곳**: `HOT_LINE`과 `TOMORROW_HOTTER`를 바꿔 보세요.

In [ ]:
HOT_LINE = 26.0          # ← 이 숫자를 바꿔 보세요 (이 온도를 넘으면 덥다고 봅니다)
TOMORROW_HOTTER = 2.0    # ← 이 숫자를 바꿔 보세요 (내일은 오늘보다 몇 도 더 더운가)

def twin(controller):
    temps, w, a = classroom(TWIN["wall_loss"], TWIN["people_heat"], controller, outdoor(TOMORROW_HOTTER))
    return temps[:, 0], w, a

def report(temps, ac):
    avg = temps.mean(axis=1)
    return {"더운 시간(분)": int(((avg > HOT_LINE) & OCCUPIED).sum()),
            "최고 온도(도)": round(avg.max(), 1),
            "에어컨 켠 시간(분)": int(np.sum(ac))}

plans = {
    "do nothing":            lambda m, T: (0, 0),
    "window open all day":   lambda m, T: (1, 0),
    "window open 9-11 only": lambda m, T: (int(CLOCK[m] < 11), 0),
    "AC during classes":     lambda m, T: (0, int(OCCUPIED[m])),
    "window open + AC":      lambda m, T: (1, int(OCCUPIED[m])),
}
names_ko = ["아무것도 안 함", "하루 종일 창문 열기", "아침에만 창문 열기", "수업 중 에어컨", "창문 열고 에어컨"]
rows, curves = [], {}
for (name, plan), ko in zip(plans.items(), names_ko):
    temps, w, a = twin(plan)
    curves[name] = temps.mean(axis=1)
    rows.append({"방법": ko, **report(temps, a)})
display(pd.DataFrame(rows))

plt.figure(figsize=(9, 4))
for name, c in curves.items():
    plt.plot(CLOCK, c, label=name)
plt.plot(CLOCK, outdoor(TOMORROW_HOTTER), "k--", lw=1, label="outdoor (forecast)")
plt.axhline(HOT_LINE, color="red", ls=":", label="hot line")
plt.xlabel("time of day (hour)"); plt.ylabel("average room temperature (C)")
plt.title("What-if tests in the virtual classroom"); plt.legend(fontsize=8, ncol=2); plt.show()

**읽는 법**: 창문을 여는 것은 바깥이 교실보다 시원할 때만 도움이 됩니다. 한낮에 바깥이 더 더우면 창문으로 오히려 더운 바람이 들어옵니다. 창문을 열고 에어컨을 켜면 시원한 공기가 새어 나갑니다. 이런 일을 진짜 교실에서 하나하나 해 봤다면 하루에 한 가지밖에 시험하지 못했을 것입니다.

## 4단계. 에어컨을 켜는 규칙 고르기

이번에는 '언제 켤지'를 정하는 **규칙**을 고릅니다. 규칙은 이렇습니다.

> 교실 평균 온도가 **켜는 온도**를 넘으면 에어컨을 켜고, 그보다 1도 낮아지면 끈다. (수업 시간에만)
> 그리고 '아침에 바깥이 교실보다 시원하면 창문을 연다'를 함께 쓸지 정한다.

켜는 온도를 24도부터 29도까지 바꿔 가며, 가상 교실에서 하루씩 돌려 봅니다. 좋은 규칙은 '더운 시간'도 짧고 '에어컨 켠 시간'도 짧아야 합니다. 둘을 하나의 점수로 합칩니다.

👉 **바꿔 볼 곳**: `ENERGY_WEIGHT`를 바꿔 보세요. 클수록 전기 아끼기를, 작을수록 시원함을 더 중요하게 봅니다.

In [ ]:
ENERGY_WEIGHT = 0.5   # ← 이 숫자를 바꿔 보세요 (에어컨 1분이 '더운 시간' 몇 분만큼 나쁜가)

def make_rule(on_temp, morning_window):
    state = {"ac": 0}
    def rule(m, T):
        avg = T.mean()
        if not OCCUPIED[m]:
            state["ac"] = 0
        elif avg > on_temp:
            state["ac"] = 1
        elif avg < on_temp - 1:
            state["ac"] = 0
        win = int(morning_window and outdoor(TOMORROW_HOTTER)[m] < avg and CLOCK[m] < 12)
        return win, state["ac"]
    return rule

rows = []
for on_temp in [24, 25, 26, 27, 28, 29]:
    for mw in [False, True]:
        temps, w, a = twin(make_rule(on_temp, mw))
        r = report(temps, a)
        r_score = r["더운 시간(분)"] + ENERGY_WEIGHT * r["에어컨 켠 시간(분)"]
        rows.append({"켜는 온도": on_temp, "아침 창문": "예" if mw else "아니오", **r, "점수(낮을수록 좋음)": round(r_score, 1)})
rules = pd.DataFrame(rows).sort_values("점수(낮을수록 좋음)")
display(rules.head(6))
CHOSEN = rules.iloc[0]
print(f"가상 교실이 고른 규칙: {CHOSEN['켜는 온도']}도를 넘으면 에어컨 켜기, 아침 창문 열기 = {CHOSEN['아침 창문']}")

## 5단계. 디지털 트윈 완성: 가상의 판단으로 진짜 교실 움직이기

지금까지 가상 교실은 진짜 교실을 따라 하기만 했습니다. 이제 방향을 바꿉니다. 내일 진짜 교실의 센서 값이 1분마다 들어오면, 4단계에서 고른 규칙이 그 값을 보고 **진짜 교실의 에어컨과 창문을 움직입니다**. 데이터가 양쪽으로 오가는 순간, 그림자는 **디지털 트윈**이 됩니다.

가상 교실이 예측한 결과와 진짜 교실에서 일어난 결과를 비교하고, 원래 습관(13~17시 에어컨)과도 비교합니다.

In [ ]:
chosen_rule = make_rule(int(CHOSEN["켜는 온도"]), CHOSEN["아침 창문"] == "예")
pred_temps, _, pred_ac = twin(make_rule(int(CHOSEN["켜는 온도"]), CHOSEN["아침 창문"] == "예"))

def sensing_rule(m, T):                  # 진짜 교실에서는 센서 값(오차 포함)을 보고 판단합니다
    noisy = T + np.random.default_rng(m).normal(0, 0.15, T.shape)
    return chosen_rule(m, noisy)

real_twin, real_w, real_ac = real_classroom(sensing_rule, hot=TOMORROW_HOTTER, seed=2)
real_old, _, old_ac = real_classroom(old_habit, hot=TOMORROW_HOTTER, seed=2)

table = pd.DataFrame([
    {"경우": "원래 습관 (진짜 교실)", **report(real_old, old_ac)},
    {"경우": "고른 규칙 - 가상 교실의 예측", **report(pred_temps, pred_ac)},
    {"경우": "고른 규칙 - 진짜 교실의 결과", **report(real_twin, real_ac)},
])
display(table)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(CLOCK, real_old.mean(1), label="old habit (real)", color="gray")
ax.plot(CLOCK, pred_temps.mean(1), "--", label="chosen rule, twin's prediction", color="tab:orange")
ax.plot(CLOCK, real_twin.mean(1), label="chosen rule, real classroom", color="tab:blue")
ax.fill_between(CLOCK, 23, 36, where=real_ac > 0, color="tab:blue", alpha=0.08, label="AC on (real)")
ax.axhline(HOT_LINE, color="red", ls=":"); ax.set_ylim(23, 36)
ax.set_xlabel("time of day (hour)"); ax.set_ylabel("average room temperature (C)")
ax.set_title("The twin's decision now drives the real classroom"); ax.legend(fontsize=8); plt.show()

**읽는 법**: 가상 교실의 예측(점선)과 진짜 교실의 결과(파란 선)가 비슷하면, 쌍둥이를 믿고 다음 결정도 맡길 수 있습니다. 둘이 많이 어긋나면 2단계로 돌아가 쌍둥이를 다시 맞춰야 합니다. 진짜 디지털 트윈은 이 '맞추기 → 시험하기 → 적용하기 → 다시 맞추기'를 쉬지 않고 되풀이합니다.

### 정리: 우리가 만든 세 가지

| 단계 | 이름 | 데이터가 오가는 방향 |
|---|---|---|
| 2단계의 계산식(손으로 숫자를 넣어 돌림) | 디지털 모델 | 사람 → 가상 |
| 1단계의 교실 그림 | 디지털 그림자 | 진짜 → 가상 (자동) |
| 5단계의 자동 제어 | 디지털 트윈 | 진짜 ⇄ 가상 (양쪽) |

## 더 해 보기

- 3단계의 `TOMORROW_HOTTER`를 `5`로 올리고 4~5단계를 다시 실행해 보세요. 고른 규칙이 바뀝니까?
- 0단계의 `_TRUE`에서 `people_heat`를 `0.018`로 바꾸면 '학생이 더 많아진 교실'이 됩니다. 2단계부터 다시 실행하지 **않고** 5단계만 실행하면, 쌍둥이의 예측과 진짜 결과가 어떻게 어긋납니까? 오래된 쌍둥이가 왜 위험한지 생각해 보세요.

## 생각해 볼 것

1. 이 실습의 쌍둥이는 온도만 봤습니다. 진짜 교실의 디지털 트윈이라면 무엇을 더 재야 할까요(사람 수, 이산화탄소, 습도 등)? 센서를 늘릴 때 생기는 좋은 점과 걱정되는 점(사생활)을 함께 쓰세요.
2. 4단계에서 `ENERGY_WEIGHT`에 따라 '좋은 규칙'이 달라졌습니다. 이 숫자는 누가 정해야 할까요? 학생, 선생님, 학교 행정실, 전기 요금을 내는 사람의 입장을 비교해 보세요.
3. 책의 '도시 디지털 트윈' 토론처럼, 우리 학교 전체를 디지털 트윈으로 만들어 냉난방, 출입, 식당 줄까지 자동으로 조절한다고 합시다. 쌍둥이가 틀렸을 때 생기는 문제와, 그것을 막을 방법을 하나씩 쓰세요.